Hackathon Script by Abazaj, Altieri, Ly, and Todesco
====================================

In [ ]:
!pip install mediacloud newspaper3k pandas lxml_html_clean

import mediacloud.api
from importlib.metadata import version
import random
import time
import datetime
import json
import os
import pandas as pd
from newspaper import Article, Config
from google.colab import drive

# Mount Drive for persistent storage
drive.mount('/content/drive')

# ── 1. API AND SCRAPING FUNCTION
MC_API_KEY = "REDACTED"
directory  = mediacloud.api.DirectoryApi(MC_API_KEY)
search_api = mediacloud.api.SearchApi(MC_API_KEY)

print(f'Media Cloud client v{version("mediacloud")}')

# ── 2. CONFIGURATIONS
QUINTILE_COLLECTION_IDS = {
    'Q1': 231013063,
    'Q2': 231013089,
    'Q4': 231013109,
    'Q5': 231013110,
}

#US_NATIONAL_COLLECTION = 34394850 # Didn't use this in the end --- we don't only want top US news, we want news Americans (Usonians) are exposed to, and that is already how the quintiles were set.
QUERY = '("climate change" OR "global warming" OR "climate crisis" OR "climate action" OR "climate emergency")'
YEARS = list(range(2016, 2018))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Media Cloud client v5.0.0


In [ ]:
# ── 3. SAFE SAVE LOGIC (AI-generated)
SEEN_IDS_FILE = "/content/drive/MyDrive/seen_ids.json"
OUTPUT_FILE = '/content/drive/MyDrive/earth_day_real_articles.json'
api_retry_backoff = [5, 15, 30]

def atomic_write(obj, path):
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f)
    os.replace(tmp, path)

def load_json(path, default):
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return default

seen_ids = set(load_json(SEEN_IDS_FILE, []))
all_records = load_json(OUTPUT_FILE, [])

# Re-populate seen_ids based on what's already in all_records to ensure consistency
for rec in all_records:
    if "url" in rec:
        seen_ids.add(rec["url"])

In [ ]:
random.seed(101)

# ── 5. MAIN COLLECTION LOOP
stories_per_per_group = {
    "Q1_22_April_2016" : None,
    "Q5_22_April_2016" : None,
    "Q1_22_April_2017" : None,
    "Q5_22_April_2017" : None,
} # These are just placeholders to visually see how the dictionary will be structured

for year in YEARS:

  for quintile in QUINTILE_COLLECTION_IDS.items():

    start_date = datetime.date(year, 4, 22)
    end_date = start_date
    count = search_api.story_count(QUERY, start_date, end_date, collection_ids=[quintile[1]])

    print(f"{quintile[0]}, 22 April {year}: {count["relevant"]} relevant articles")
    this_day_this_quintile = search_api.story_sample(
                                                      query=QUERY,
                                                      start_date=start_date,
                                                      end_date=end_date,
                                                      collection_ids=[quintile[1]],
                                                      limit=1000
                                                  )

    #assert 250 == len(this_day_this_quintile), f"Whoops! Not the right number of articles for 22 April {year} and quintile {quintile[0]}! The number is {len(this_day_this_quintile)}."
    stories_per_per_group[f"{quintile[0]}_22_April_{year}"] = this_day_this_quintile

for item in stories_per_per_group.items():
  print(f"For {item[0]}, we have {len(item[1])} articles. Here are the first 10:")
  print(item[1][:10])

Q1, 22 April 2016: 28 relevant articles
Q2, 22 April 2016: 250 relevant articles
Q4, 22 April 2016: 30 relevant articles
Q5, 22 April 2016: 40 relevant articles
Q1, 22 April 2017: 99 relevant articles
Q2, 22 April 2017: 267 relevant articles
Q4, 22 April 2017: 88 relevant articles
Q5, 22 April 2017: 51 relevant articles
For Q1_22_April_2016, we have 28 articles. Here are the first 10:
[{'id': '914a5362065f24ce5fb8b0df2bc7e4693a968576f90b945333b12ed0b30316e9', 'indexed_date': datetime.datetime(2025, 7, 17, 11, 34, 40, 575944, tzinfo=datetime.timezone.utc), 'language': 'en', 'media_name': 'aljazeera.com', 'media_url': 'aljazeera.com', 'publish_date': datetime.date(2016, 4, 22), 'title': 'Palestine: A distinctive voice for climate action', 'url': 'http://www.aljazeera.com/indepth/opinion/2016/04/palestine-distinctive-voice-climate-action-160422170052768.html'}, {'id': 'b4f46ff0513805aaca95045a7b6a555a6d413743c8d35c5ae6187d8424846b47', 'indexed_date': datetime.datetime(2024, 11, 19, 14, 29

In [ ]:
# ── 6. EXPLORING REPLICAS

articles_with_replicas = 0
for item in stories_per_per_group.items():
  articles_with_replicas += len(item[1])
print(f"We are starting off with {articles_with_replicas} articles, including replicas.")
print()

print("Filtering, IGNORING any replicas across groups for now...")
stories_per_per_group_clean = {}

non_replicated_articles = []
replicated_articles_filtered_out = []

for item in stories_per_per_group.items():
  this_day_this_quintile_clean = [] # For the full article data
  this_day_this_quintile_clean_titles = [] # For just the article titles
  for article in item[1]:
    if article['title'] not in this_day_this_quintile_clean_titles:
      non_replicated_articles.append(article)
      this_day_this_quintile_clean_titles.append(article['title'])
      this_day_this_quintile_clean.append(article)
    else:
      replicated_articles_filtered_out.append(article)
  stories_per_per_group_clean[item[0]] = this_day_this_quintile_clean
print(f"We filtered out {len(replicated_articles_filtered_out)} replicas.")

print(f"There are now {articles_with_replicas} - {len(replicated_articles_filtered_out)} = {len(non_replicated_articles)} articles after filtering.")
print("These articles are sorted as follows:")
for item in stories_per_per_group_clean.items():
  print(f"For {item[0]}, we have {len(item[1])} articles")
print()

print(f"How many replicated articles are there across the {len(stories_per_per_group_clean.items())} groups now?")
check_list = []
replicated_articles_across_groups = []
for article in non_replicated_articles:
  if article['title'] in check_list:
    replicated_articles_across_groups.append(article)
  else:
    check_list.append(article['title'])
print(f"Answer: there are still {len(replicated_articles_across_groups)} replicas across these groups. We had only taken out the replicas within groups.")
print("But why do these replicas across groups even exist?")
print()

titles_of_replicas_across_groups = []
for article in replicated_articles_across_groups:
  titles_of_replicas_across_groups.append(article['title'])

replicated_articles_overview = {}
for group in stories_per_per_group_clean.items():
  for article in group[1]:
    if article['title'] in titles_of_replicas_across_groups:
      entry = (group[0], article)
      if article['title'] in replicated_articles_overview:
        replicated_articles_overview[article['title']].append(entry)
      else:
        replicated_articles_overview[article['title']] = [entry]


for unique_title in replicated_articles_overview.keys():
  print(f"Here are all the instances, across groups, of what is detected as an article titled \"{unique_title}\":")
  for group_name, article in replicated_articles_overview[unique_title]:
    print(f"  [{group_name}] {article}")
  print()

We are starting off with 853 articles, including replicas.

Filtering, IGNORING any replicas across groups for now...
We filtered out 255 replicas.
There are now 853 - 255 = 598 articles after filtering.
These articles are sorted as follows:
For Q1_22_April_2016, we have 28 articles
For Q5_22_April_2016, we have 38 articles
For Q1_22_April_2017, we have 76 articles
For Q5_22_April_2017, we have 42 articles
For Q2_22_April_2016, we have 166 articles
For Q4_22_April_2016, we have 27 articles
For Q2_22_April_2017, we have 174 articles
For Q4_22_April_2017, we have 47 articles

How many replicated articles are there across the 8 groups now?
Answer: there are still 66 replicas across these groups. We had only taken out the replicas within groups.
But why do these replicas across groups even exist?

Here are all the instances, across groups, of what is detected as an article titled "Palestine: A distinctive voice for climate action":
  [Q1_22_April_2016] {'id': '914a5362065f24ce5fb8b0df2bc7e

In [ ]:
# ── 7. REMOVING REPLICAS

titles_with_inter_group_replicas = replicated_articles_overview.keys()
groups = stories_per_per_group_clean.keys()

for group in groups:
    before = len(stories_per_per_group_clean[group])
    stories_per_per_group_clean[group] = [
        article for article in stories_per_per_group_clean[group]
        if article['title'] not in titles_with_inter_group_replicas
    ]
    after = len(stories_per_per_group_clean[group])
    print(f"{group}: removed {before - after} articles, {after} remaining.")

Q1_22_April_2016: removed 9 articles, 19 remaining.
Q5_22_April_2016: removed 3 articles, 35 remaining.
Q1_22_April_2017: removed 32 articles, 44 remaining.
Q5_22_April_2017: removed 9 articles, 33 remaining.
Q2_22_April_2016: removed 7 articles, 159 remaining.
Q4_22_April_2016: removed 9 articles, 18 remaining.
Q2_22_April_2017: removed 33 articles, 141 remaining.
Q4_22_April_2017: removed 20 articles, 27 remaining.


In [ ]:
print("With the max amount of filtering done:")
print()
groups = list(stories_per_per_group_clean.keys())
groups.sort()
count = 0
for group in groups:
  print(f"{group} has {len(stories_per_per_group_clean[group])} articles")
  count += 1
  if count == 4:
    print()

print()
print()
print("Therefore:")
print()
print(f"Leftwing, 2016: {len(stories_per_per_group_clean['Q1_22_April_2016'])} + {len(stories_per_per_group_clean['Q2_22_April_2016'])} = {len(stories_per_per_group_clean['Q1_22_April_2016']) + len(stories_per_per_group_clean['Q2_22_April_2016'])} articles")
print(f"Leftwing, 2017: {len(stories_per_per_group_clean['Q1_22_April_2017'])} + {len(stories_per_per_group_clean['Q2_22_April_2017'])} = {len(stories_per_per_group_clean['Q1_22_April_2017']) + len(stories_per_per_group_clean['Q2_22_April_2017'])} articles")
print()
print(f"Rightwing, 2016: {len(stories_per_per_group_clean['Q4_22_April_2016'])} + {len(stories_per_per_group_clean['Q5_22_April_2016'])} = {len(stories_per_per_group_clean['Q4_22_April_2016']) + len(stories_per_per_group_clean['Q5_22_April_2016'])} articles")
print(f"Rightwing, 2017: {len(stories_per_per_group_clean['Q4_22_April_2017'])} + {len(stories_per_per_group_clean['Q5_22_April_2017'])} = {len(stories_per_per_group_clean['Q4_22_April_2017']) + len(stories_per_per_group_clean['Q5_22_April_2017'])} articles")

With the max amount of filtering done:

Q1_22_April_2016 has 19 articles
Q1_22_April_2017 has 44 articles
Q2_22_April_2016 has 159 articles
Q2_22_April_2017 has 141 articles

Q4_22_April_2016 has 18 articles
Q4_22_April_2017 has 27 articles
Q5_22_April_2016 has 35 articles
Q5_22_April_2017 has 33 articles


Therefore:

Leftwing, 2016: 19 + 159 = 178 articles
Leftwing, 2017: 44 + 141 = 185 articles

Rightwing, 2016: 18 + 35 = 53 articles
Rightwing, 2017: 27 + 33 = 60 articles


In [ ]:
# ── 8. SAVING TO JSON (AI-generated)
import datetime

class DateTimeEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, (datetime.date, datetime.datetime)):
            return obj.isoformat()
        return super().default(obj)

def atomic_write(obj, path):
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, cls=DateTimeEncoder)
    os.replace(tmp, path)

atomic_write(stories_per_per_group_clean, OUTPUT_FILE)